In [ ]:
import csv, json, zipfile, urllib.request, base64
from pathlib import Path
import numpy as np, pandas as pd, torch, timm
from torchvision import transforms
from PIL import Image
from sklearn.linear_model import LogisticRegression
from google.colab import files
W=Path("/content")
url="https://gist.githubusercontent.com/Quadrotez/0b63743717326adc1b3022cd372cc94d/raw/inv_b64.txt"
urllib.request.urlretrieve(url, W/"b64.txt")
(W/"inv.zip").write_bytes(base64.b64decode((W/"b64.txt").read_bytes()))
with zipfile.ZipFile(W/"inv.zip") as z: z.extractall(W/"d")
D=W/"d"
lab=pd.read_csv(D/"train_labels.csv"); oof=pd.read_csv(D/"oof.csv"); test=pd.read_csv(D/"test.csv")
MATS=["metal","wood","plastic","glass","paper","mineral"]
dev="cuda" if torch.cuda.is_available() else "cpu"
print("device",dev,len(lab),len(oof),len(test),flush=True)
m=timm.create_model("vit_base_patch16_224.dino",pretrained=True,num_classes=0).to(dev).eval()
t=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize((.5,.5,.5),(.5,.5,.5))])
def emb(df):
    out = []
    with torch.no_grad():
        for i in range(0, len(df), 32):
            keys = df['key'].iloc[i:i + 32]
            ims = [t(Image.fromarray(np.load(D / (k + '.npy'))[:3].transpose(1, 2, 0)))
                   for k in keys]
            out.append(m(torch.stack(ims).to(dev)).float().cpu().numpy())
    return np.concatenate(out)
EF=emb(lab); ET=emb(test)
print("emb",EF.shape,ET.shape,flush=True)
oof_p=np.zeros((len(oof),7)); tb=np.zeros(len(test)); tm=np.zeros((len(test),6))
pos={(r.image_id,int(r.hyp_idx)):i for i,r in enumerate(lab.itertuples())}
for fold in range(1,6):
    tr=lab[(lab.fold!=fold)&lab.material.isin(MATS)]
    idx=[pos[(r.image_id,int(r.hyp_idx))] for r in oof.itertuples() if int(r.fold)==fold and (r.image_id,int(r.hyp_idx)) in pos]
    fb=LogisticRegression(C=1,class_weight="balanced",max_iter=2000).fit(EF[tr.index.to_numpy()],tr.form)
    fm=LogisticRegression(C=1,class_weight="balanced",max_iter=2000).fit(EF[tr.index.to_numpy()],tr.material)
    for i in idx:
        oof_p[i,0]=fb.predict_proba(EF[i:i+1])[0,list(fb.classes_).index("bulk")]
        pr=fm.predict_proba(EF[i:i+1])[0]; oof_p[i,1:]=[pr[list(fm.classes_).index(x)] if x in fm.classes_ else 0 for x in MATS]
    tb+=fb.predict_proba(ET)[:,list(fb.classes_).index("bulk")]/5
    pr=fm.predict_proba(ET); tm+=np.column_stack([pr[:,list(fm.classes_).index(x)] if x in fm.classes_ else np.zeros(len(ET)) for x in MATS])/5
    print("fold",fold,flush=True)
with (W/"oof_probs.csv").open("w",newline="") as fh:
    w=csv.writer(fh); w.writerow(["image_id","hyp_idx","fold","detection_sha256","p_bulk"]+[f"p_material_{x}" for x in MATS])
    for i,r in enumerate(oof.itertuples()): w.writerow([r.image_id,r.hyp_idx,r.fold,r.detection_sha256,f"{oof_p[i,0]:.8f}"]+[f"{v:.8f}" for v in oof_p[i,1:]])
with (W/"test_probs.csv").open("w",newline="") as fh:
    w=csv.writer(fh); w.writerow(["image_id","hyp_idx","detection_sha256","p_bulk"]+[f"p_material_{x}" for x in MATS])
    for i,r in enumerate(test.itertuples()): w.writerow([r.image_id,r.hyp_idx,r.detection_sha256,f"{tb[i]:.8f}"]+[f"{v:.8f}" for v in tm[i]])
print("SAVED",flush=True)
files.download(str(W/"oof_probs.csv")); files.download(str(W/"test_probs.csv"))
